# Method: Multilevel Perceptron

### I. Imports and Reads

#### Imports

In [12]:
import numpy as np
import csv 

#### Reads

In [13]:
with open('../dataset/processed_test.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    test_in = np.array(data, dtype=np.float64)
    test_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_tune.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    tune_in = np.array(data, dtype=np.float64)
    tune_out = np.array(output, dtype=np.float64)

with open('../dataset/processed_train.csv', 'r') as f:
    read = csv.reader(f)
    data = []
    output = []
    for x in read:
        data.append(x[:-1])
        output.append(x[-1])

    train_in = np.array(data, dtype=np.float64)
    train_out = np.array(output, dtype=np.float64)

### II. Setup

#### Initial Parameters


In [ ]:
learning_rate = 0.0001
input_size = train_in.shape[1]
N = train_in.shape[0]
batch_size = N
layer_sizes = [16, 16, 16] + [1]


#### Activation Functions

In [15]:
def ReLU(layers):
    if type(layers) == list:
        layers = np.array(layers, dtype=np.float64)
    return np.where(layers < 0, 0, layers)

def ReLUDerivative(layers):
    return np.where(layers < 0, 0, 1)

#### Layer Setup

In [16]:
weights = [np.zeros((input_size, layer_sizes[0]), dtype=np.float64)]
biases = []
layers = [np.zeros((batch_size, input_size), dtype=np.float64)]
for x in layer_sizes:
    biases.append(np.ones((x), dtype=np.float64))
    layers.append(np.zeros((batch_size, x), dtype=np.float64))

for i in range(0, len(layer_sizes)-1):
    weights.append(np.zeros((layer_sizes[i], layer_sizes[i+1]), dtype=np.float64))


print([x.shape for x in layers])
print([x.shape for x in weights])
print([x.shape for x in biases])

[(3174, 6), (3174, 16), (3174, 32), (3174, 16), (3174, 1)]
[(6, 16), (16, 32), (32, 16), (16, 1)]
[(16,), (32,), (16,), (1,)]


### III. Training

In [17]:
iters = 500000
learning_rate = 0.00001
for iter in range(iters):

    # get batch_size datapoints
    cost = np.zeros((batch_size))
    indexes = np.random.choice(range(N), size = batch_size, replace= False)
    layers[0] = train_in[indexes]
    expected = train_out[indexes].reshape(-1, 1)

    #forward prop
    for i in range(len(layers)-1):
        if i < len(layers)-1: 
            layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
        else:
            layers[i+1] = layers[i].dot(weights[i])+biases[i]

    cost = (layers[-1] - expected)**2

    # print(cost)
    print(f"Mean Squared Error [{iter}]: {cost.mean()}")
    # backward prop

    bias_diff = [np.zeros_like(x) for x in biases]
    weights_diff = [np.zeros_like(x) for x in weights]

    delta = 2 * (layers[-1] - expected)
    for i in range(len(layer_sizes) - 1, -1, -1):
        weights_diff[i] = layers[i].T @ delta / batch_size
        bias_diff[i] = delta.mean(axis = 0)

        if i > 0:
            delta = delta @ weights[i].T * ReLUDerivative(layers[i])

    for x in range(len(layer_sizes)-1, -1, -1):
            weights[x] = weights[x] - learning_rate * weights_diff[x]
            biases[x] = biases[x] - learning_rate * bias_diff[x]


Mean Squared Error [0]: 1662.562158979206
Mean Squared Error [1]: 1661.4536181037286
Mean Squared Error [2]: 1660.3457860436629
Mean Squared Error [3]: 1659.2385727625467
Mean Squared Error [4]: 1658.1318885515539
Mean Squared Error [5]: 1657.0256440211595
Mean Squared Error [6]: 1655.9197500929165
Mean Squared Error [7]: 1654.8141179913462
Mean Squared Error [8]: 1653.7086592359406
Mean Squared Error [9]: 1652.6032856332763
Mean Squared Error [10]: 1651.4979092692283
Mean Squared Error [11]: 1650.3924425012938
Mean Squared Error [12]: 1649.286797951015
Mean Squared Error [13]: 1648.180888496502
Mean Squared Error [14]: 1647.0746272650513
Mean Squared Error [15]: 1645.9679276258657
Mean Squared Error [16]: 1644.8607031828612
Mean Squared Error [17]: 1643.7528677675632
Mean Squared Error [18]: 1642.6443354321025
Mean Squared Error [19]: 1641.535020442288
Mean Squared Error [20]: 1640.424837270769
Mean Squared Error [21]: 1639.3137005902827
Mean Squared Error [22]: 1638.201525266979
Mean

### IV. Test Weights on a Dataset

In [18]:
layers[0] = test_in
expected = test_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2


SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()

print(f"RMSE [test]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [test]: {(layers[-1] - expected).mean()}")
print(f"MAPE [test]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [test]: {1 - (SSres/SStotal)}")


# get batch_size datapoints
layers[0] = tune_in
expected = tune_out.reshape(-1, 1)

#forward prop
for i in range(len(layers)-1):
    if i < len(layers)-1: 
        layers[i+1] = ReLU(layers[i].dot(weights[i])+biases[i])
    else:
        layers[i+1] = layers[i].dot(weights[i])+biases[i]

cost = (layers[-1] - expected)**2

# print(cost)
SStotal = (expected - expected.mean())**2
SStotal = SStotal.sum()

SSres = (layers[-1] - expected)**2
SSres = SSres.sum()


print(f"\nRMSE [tune]: {((layers[-1] - expected)**2).mean() ** 0.5}")
print(f"MAE [tune]: {(layers[-1] - expected).mean()}")
print(f"MAPE [tune]: {np.abs((expected - layers[-1])/expected).mean()*100}%")
print(f"R2 [tune]: {1 - (SSres/SStotal)}")

RMSE [test]: 2.5914303728834396
MAE [test]: 0.6975372052209492
MAPE [test]: 4.367201858712804%
R2 [test]: 0.7178659635220661

RMSE [tune]: 2.263239481297372
MAE [tune]: 0.9689128645353132
MAPE [tune]: 4.335988150240529%
R2 [tune]: 0.7489684030650454


### V. Save current weights

## DONT FORGET TO RE-COMMENT AFTER SAVING TO PREVENT LOSS OF DATA

In [19]:
# np.savez("MLP_weights", *weights)
# np.savez("MLP_biases", *biases)